# Issue #225 — Normal-junction filter strength on patient_001 (chr22)

**Issue:** [#225](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/225)  
**Parent:** [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203) (Experiment 3)  
**Design spec:** [docs/superpowers/specs/2026-05-21-issue-225-normal-junction-filter-strength-design.md](../../../docs/superpowers/specs/2026-05-21-issue-225-normal-junction-filter-strength-design.md)

Compare three normal-junction filter sources on patient_001's chr22 tumor junctions: matched-normal, Snaptron-chr22-GTEx-proxy, and AlphaGenome-predicted-normal. Produce the decision-rule numbers for the Exp 3 row of [Issue #203](https://github.com/Jin-HoMLee/splice-neoepitope-pipeline/issues/203)'s decision table.

**Scope:** chr22 only (test-config harness); patient_001 only; matching key `(chrom, donor_0based, acceptor_0based_excl, strand)`, exact match.

## §0 — Setup + path guards

In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "workflow").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENT_DIR = REPO_ROOT / "research" / "experiments" / "issue_225_normal_junction_filter_strength"
OUTPUTS_DIR = EXPERIMENT_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHROM = "chr22"

# Inputs (referenced by path; not copied)
TUMOR_TSV = REPO_ROOT / "results" / "patient_001_test" / "alignment" / "SRR9143066_test" / "junctions.tsv"
NORMAL_TSV = REPO_ROOT / "results" / "patient_001_test" / "alignment" / "SRR9143065_test" / "junctions.tsv"
AG_PARQUET = REPO_ROOT / "research" / "notebooks" / "issue_224_alphagenome_exp1_outputs" / "chr22_stomach_predicted_junctions.parquet"

# Outputs
GTEX_PANEL_PARQUET = OUTPUTS_DIR / "chr22_gtex_panel.parquet"
OVERLAP_TSV = OUTPUTS_DIR / "filter_overlap_table.tsv"
VENN_PNG = OUTPUTS_DIR / "filter_venn_chr22.png"

# Path-existence guards
inputs = {
    "Tumor junctions": TUMOR_TSV,
    "Matched-normal junctions": NORMAL_TSV,
    "AlphaGenome predictions": AG_PARQUET,
}
missing = [(label, path) for label, path in inputs.items() if not path.exists()]
if missing:
    msg_lines = [f"  - {label}: {path}" for label, path in missing]
    fix_hint = (
        "\nTo produce the chr22 tumor junctions (the most common miss), run:\n"
        "  conda activate snakemake && snakemake --cores 4 --use-conda --configfile config/test_config.yaml "
        "-- results/patient_001_test/alignment/SRR9143066_test/junctions.tsv"
    )
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(msg_lines) + fix_hint)

print(f"Repo root:      {REPO_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR.relative_to(REPO_ROOT)}")
print(f"All inputs present: {all(p.exists() for p in inputs.values())}")

## §1 — Tumor junctions (patient_001 chr22)

Load the post-PR #372 `junctions.tsv` produced by `workflow/scripts/bed12_to_junctions.py`. Format: 2-column TSV, no header: `<chrom>:<donor_1based>:<acceptor_0based_excl>:<strand>\t<reads>`. Loader normalizes to **0-based half-open intron coords** for set-ops downstream.

In [ ]:
def load_pipeline_junctions(tsv_path: Path) -> pd.DataFrame:
    """Load 2-column junctions TSV from bed12_to_junctions.py / STAR awk.

    Format: ``<chrom>:<donor_1based>:<acceptor_0based_excl>:<strand>\\t<reads>``.
    Returns DataFrame with 0-based half-open intron coords for set operations.
    """
    raw = pd.read_csv(tsv_path, sep="\t", header=None, names=["key", "count"])
    parts = raw["key"].str.split(":", expand=True)
    parts.columns = ["chrom", "donor_1based", "acceptor_0based_excl", "strand"]
    return pd.DataFrame({
        "chrom": parts["chrom"],
        "donor": parts["donor_1based"].astype(int) - 1,
        "acceptor": parts["acceptor_0based_excl"].astype(int),
        "strand": parts["strand"],
        "count": raw["count"].astype(int),
    })

tumor = load_pipeline_junctions(TUMOR_TSV)
assert (tumor["chrom"] == TARGET_CHROM).all(), f"Expected {TARGET_CHROM}-only tumor; got {tumor['chrom'].unique()}"
assert len(tumor) > 0, "Tumor junctions empty — check alignment for SRR9143066"
print(f"Tumor junctions (chr22):     {len(tumor):,}")
print(f"Read-count summary: {tumor['count'].describe().to_dict()}")
tumor.head()

## §2 — Filter sets

### §2(a) — Matched-normal (current pipeline source)

Load matched-normal junctions from the same `junctions.tsv` format. Anchor of the comparison; gold standard for tissue-expressed splicing in patient_001's stomach normal.

In [ ]:
mn = load_pipeline_junctions(NORMAL_TSV)
assert (mn["chrom"] == TARGET_CHROM).all()
assert len(mn) > 0

# Build the matching-key set: (chrom, donor, acceptor, strand)
def to_key_set(df: pd.DataFrame) -> set[tuple]:
    return set(map(tuple, df[["chrom", "donor", "acceptor", "strand"]].itertuples(index=False, name=None)))

mn_set = to_key_set(mn)
print(f"Matched-normal junctions:    {len(mn):,}  (unique keys: {len(mn_set):,})")